# Archived Development Notebook

**Status:** Archived exploratory analysis

This notebook contains exploratory analyses, debugging investigations, preprocessing validation, or development experiments performed during construction of the Raman spectroscopy bacterial classification framework.

The analyses documented here supported repository development but are **not part of the finalized methodology** described in the README, documentation, or accompanying publication.

This notebook is preserved for transparency and reproducibility.

In [ ]:
import numpy as np
import pandas as pd

from pathlib import Path

# =========================================================
# PATHS
# =========================================================

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data" / "raw"

# =========================================================
# ALL DATA FILES
# =========================================================

all_files = sorted(DATA_DIR.glob("*.npy"))

files_df = pd.DataFrame({
    "file_name": [f.name for f in all_files],
    "path": [str(f) for f in all_files]
})

files_df

,file_name,path
0,wavenumbers.npy,s:\raman-spectral-classifier\data\raw\wavenumb...
1,X_2018clinical.npy,s:\raman-spectral-classifier\data\raw\X_2018cl...
2,X_2019clinical.npy,s:\raman-spectral-classifier\data\raw\X_2019cl...
3,X_finetune.npy,s:\raman-spectral-classifier\data\raw\X_finetu...
4,X_reference.npy,s:\raman-spectral-classifier\data\raw\X_refere...
5,X_test.npy,s:\raman-spectral-classifier\data\raw\X_test.npy
6,y_2018clinical.npy,s:\raman-spectral-classifier\data\raw\y_2018cl...
7,y_2019clinical.npy,s:\raman-spectral-classifier\data\raw\y_2019cl...
8,y_finetune.npy,s:\raman-spectral-classifier\data\raw\y_finetu...
9,y_reference.npy,s:\raman-spectral-classifier\data\raw\y_refere...


In [4]:
inspection_rows = []

for file_path in all_files:

    arr = np.load(file_path)

    row = {
        "file_name": file_path.name,
        "shape": arr.shape,
        "dtype": arr.dtype,
        "min": np.min(arr),
        "max": np.max(arr)
    }

    if file_path.name.startswith("y_"):

        unique_vals = np.unique(arr)

        row["n_unique"] = len(unique_vals)
        row["unique_values"] = unique_vals.tolist()

    else:

        row["n_unique"] = "-"
        row["unique_values"] = "-"

    inspection_rows.append(row)

inspection_df = pd.DataFrame(inspection_rows)

inspection_df

,file_name,shape,dtype,min,max,n_unique,unique_values
0,wavenumbers.npy,"(1000,)",float64,381.98,1792.4,-,-
1,X_2018clinical.npy,"(10000, 1000)",float64,0.00,1.0,-,-
2,X_2019clinical.npy,"(2500, 1000)",float64,0.00,1.0,-,-
3,X_finetune.npy,"(3000, 1000)",float64,0.00,1.0,-,-
4,X_reference.npy,"(60000, 1000)",float64,0.00,1.0,-,-
5,X_test.npy,"(3000, 1000)",float64,0.00,1.0,-,-
6,y_2018clinical.npy,"(10000,)",float64,0.00,6.0,5,"[0.0, 2.0, 3.0, 5.0, 6.0]"
7,y_2019clinical.npy,"(2500,)",float64,0.00,6.0,5,"[0.0, 2.0, 3.0, 5.0, 6.0]"
8,y_finetune.npy,"(3000,)",float64,0.00,29.0,30,"[0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, ..."
9,y_reference.npy,"(60000,)",float64,0.00,29.0,30,"[0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, ..."


In [8]:
# =========================================================
# DATASET SEMANTIC SUMMARY
# =========================================================

summary_rows = []

split_configs = [
    ("reference", "isolate_space"),
    ("finetune", "isolate_space"),
    ("test", "isolate_space"),
    ("2018clinical", "sparse_global_treatment_space"),
    ("2019clinical", "sparse_global_treatment_space")
]

for split_name, label_space in split_configs:

    X = np.load(DATA_DIR / f"X_{split_name}.npy")
    y = np.load(DATA_DIR / f"y_{split_name}.npy")

    summary_rows.append({
        "split": split_name,
        "spectra_shape": X.shape,
        "label_shape": y.shape,
        "signal_length": X.shape[1],
        "n_samples": len(X),
        "n_classes": len(np.unique(y)),
        "label_space": label_space
    })

semantic_summary_df = pd.DataFrame(summary_rows)

semantic_summary_df

,split,spectra_shape,label_shape,signal_length,n_samples,n_classes,label_space
0,reference,"(60000, 1000)","(60000,)",1000,60000,30,isolate_space
1,finetune,"(3000, 1000)","(3000,)",1000,3000,30,isolate_space
2,test,"(3000, 1000)","(3000,)",1000,3000,30,isolate_space
3,2018clinical,"(10000, 1000)","(10000,)",1000,10000,5,sparse_global_treatment_space
4,2019clinical,"(2500, 1000)","(2500,)",1000,2500,5,sparse_global_treatment_space


In [2]:
import numpy as np
import pandas as pd

from pathlib import Path

# =========================================================
# PATHS
# =========================================================

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data" / "raw"

FILES = {
    "reference": {
        "x": DATA_DIR / "X_reference.npy",
        "y": DATA_DIR / "y_reference.npy",
        "role": "source"
    },

    "finetune": {
        "x": DATA_DIR / "X_finetune.npy",
        "y": DATA_DIR / "y_finetune.npy",
        "role": "adaptation"
    },

    "test": {
        "x": DATA_DIR / "X_test.npy",
        "y": DATA_DIR / "y_test.npy",
        "role": "holdout"
    },

    "2018clinical": {
        "x": DATA_DIR / "X_2018clinical.npy",
        "y": DATA_DIR / "y_2018clinical.npy",
        "role": "ood_eval"
    },

    "2019clinical": {
        "x": DATA_DIR / "X_2019clinical.npy",
        "y": DATA_DIR / "y_2019clinical.npy",
        "role": "ood_eval"
    }
}

# =========================================================
# DATASET SUMMARY
# =========================================================

summary_rows = []

loaded_data = {}

for split_name, split_info in FILES.items():

    X = np.load(split_info["x"])
    y = np.load(split_info["y"]).astype(int)

    loaded_data[split_name] = {
        "X": X,
        "y": y
    }

    summary_rows.append({
        "split": split_name,
        "X_shape": X.shape,
        "y_shape": y.shape,
        "n_samples": len(X),
        "signal_length": X.shape[1],
        "n_classes": len(np.unique(y)),
        "classes": sorted(np.unique(y).tolist())
    })

dataset_summary_df = pd.DataFrame(summary_rows)

dataset_summary_df

,split,X_shape,y_shape,n_samples,signal_length,n_classes,classes
0,reference,"(60000, 1000)","(60000,)",60000,1000,30,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,..."
1,finetune,"(3000, 1000)","(3000,)",3000,1000,30,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,..."
2,test,"(3000, 1000)","(3000,)",3000,1000,30,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,..."
3,2018clinical,"(10000, 1000)","(10000,)",10000,1000,5,"[0, 2, 3, 5, 6]"
4,2019clinical,"(2500, 1000)","(2500,)",2500,1000,5,"[0, 2, 3, 5, 6]"
